In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
import json
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

print("Training started...")

IMG_SIZE = (128, 128)
BATCH_SIZE = 16

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/input/datasets/medasivamanoj/leaf-disease/train1/train1",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_data = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/input/datasets/medasivamanoj/leaf-disease/validation1/validation1",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/input/datasets/medasivamanoj/leaf-disease/test1/test1",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_data.class_names
print(f"Classes: {class_names}")


normalization_layer = layers.Rescaling(1./255)

train_data = train_data.map(lambda x, y: (normalization_layer(x), y))
val_data = val_data.map(lambda x, y: (normalization_layer(x), y))
test_data = test_data.map(lambda x, y: (normalization_layer(x), y))


print("Data ready...")


model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.Conv2D(16, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(64, activation='relu',kernel_regularizer=regularizers.l2(0.001)), 

    layers.Dropout(0.4),

    layers.Dense(len(class_names), activation='softmax')
])

print("Model built...")


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled...")

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)


lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2,
    verbose=1
)


history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=19,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)


y_true = []
y_pred = []

for images, labels in val_data:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))


model.save("/kaggle/working/plant_disease_model_new.keras")

with open("/kaggle/working/class_names.json", "w") as f:
    json.dump(class_names, f)

with open("/kaggle/working/history.json", "w") as f:
    json.dump(history.history, f)

print(f"\nFinal Validation Accuracy: {history.history['val_accuracy'][-1]*100:.2f}%")

print("Training completed!")


print("\nEvaluating on Test Data...")
test_loss, test_acc = model.evaluate(test_data, verbose=1)

print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

y_true_test = []
y_pred_test = []

for images, labels in test_data:
    preds = model.predict(images, verbose=0)
    y_true_test.extend(labels.numpy())
    y_pred_test.extend(np.argmax(preds, axis=1))

print("\nTest Classification Report:")
print(classification_report(y_true_test, y_pred_test, target_names=class_names))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_true_test, y_pred_test))




2026-04-27 14:51:51.514367: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777301511.665924      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777301511.708270      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777301512.061967      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777301512.062005      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777301512.062007      16 computation_placer.cc:177] computation placer alr

Training started...
Found 27494 files belonging to 38 classes.
Found 608 files belonging to 38 classes.


2026-04-27 14:52:16.202451: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Found 7016 files belonging to 38 classes.
Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_